# **Clinical Trial Enrichment: Run 2 (Molecular Blueprint)**
---
This notebook assembles the unified context for the second stage of enrichment. It focuses on pharmacological and molecular evidence, injecting the disease mappings from Run 1 and expanding the drug, mechanism, and endpoint details.

# **0. Master Control: Reset & Configuration**
Define the execution mode, sampling parameters, and reset artifacts upfront.

In [1]:
# [CONTROL] Set to True to enable deletion of existing input contexts
RESET_PRODUCED_FILES = True

# [CONFIG] Toggle between full run and test sample
IS_PRODUCTION = True  # Set to True for full run
SAMPLE_SIZE = 1000
RANDOM_STATE = 2012

import os
files_to_delete = [
        '../data/llm_in_02.csv',
    ]

if RESET_PRODUCED_FILES:
    for f in files_to_delete:
        if os.path.exists(f):
            os.remove(f)
            print(f"> Deleted: {f}")
    print("> Reset complete.")
else:
    print("> Reset skipped.")

> Reset complete.


# **1. Environment Setup**
Initialize libraries and project-specific utilities.

In [2]:
import pandas as pd
import os
import csv
import re
import sys
import json
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()
sys.path.append('..')
from src.prep.text_cleaning import day_zero_reconstructor

DATA_PATH = '../data/'
OUTPUT_PATH = '../data/processed'
NL = chr(10)

# [STEP 5] Utility function for robust CSV loading with specific clinical formatting
def safe_load(filename, cols=None):
    full_path = os.path.join(DATA_PATH, filename)
    # Match Strategy A: PERFECT from data_loader_clinpred.py
    params_perfect = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": csv.QUOTE_MINIMAL, "low_memory": False, "on_bad_lines": "warn"
    }
    # Match Strategy B: ROBUST
    params_robust = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": 3, "low_memory": False, "on_bad_lines": "warn"
    }
    try:
        return pd.read_csv(full_path, usecols=cols, **params_perfect)
    except:
        return pd.read_csv(full_path, usecols=cols, **params_robust)

print("> SUCCESS: Environment Ready.")


> SUCCESS: Environment Ready.


# **2. Anchor Alignment & Result Ingestion**
We load the finalized Indication and Therapeutic Area assignments from Run 1. These serve as the ground-truth anchors for the molecular pass, ensuring clinical consistency across the pipeline.

In [3]:
# [STEP 1] Load results from Run 1 (the 'anchors')
df_run1 = pd.read_csv(os.path.join(OUTPUT_PATH, 'llm_out_01.csv'), usecols=['nct_id', 'gbd_indication_name', 'therapeutic_area'])
run1_lookup = df_run1.set_index('nct_id').to_dict('index')


# **3. Trial Selection & Sampling**
Based on the configuration set in Section 1, this block selects the target cohort. In production, it ensures all Run 1 anchors are included; in testing, it selects a reproducible random sample.

In [4]:
# [STEP 2] TRIAL SELECTION: Production vs. Testing
if IS_PRODUCTION:
    target_ids = df_run1['nct_id'].tolist()
    print(f"> PRODUCTION MODE: Processing all {len(target_ids)} trials.")
else:
    target_ids = df_run1['nct_id'].sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).tolist()
    print(f"> TESTING MODE: Processing {SAMPLE_SIZE} random trials.")


> PRODUCTION MODE: Processing all 29557 trials.


# **4. Core Metadata & Evidence Harvesting**
We retrieve official trial titles and basic metadata, filtering them to match our target cohort. We then perform an enhanced harvest of conditions, drug aliases, mechanisms of action, and primary endpoints.

In [5]:
# [STEP 3] Load studies metadata
df_studies = safe_load('studies.txt', cols=['nct_id', 'official_title', 'brief_title', 'start_date', 'phase'])
df_studies['official_title'] = df_studies['official_title'].fillna(df_studies['brief_title'])
studies_lookup = df_studies[df_studies['nct_id'].isin(target_ids)].set_index('nct_id').to_dict('index')

print(f"> Loaded {len(target_ids)} trials from Run 1 results.")

> Loaded 29557 trials from Run 1 results.


In [6]:
print(">>> Harvesting Enhanced Evidence...")

# [STEP 1] Conditions
df_cond = safe_load('conditions.txt', cols=['nct_id', 'name'])
cond_lookup = defaultdict(list)
for _, row in df_cond[df_cond['nct_id'].isin(target_ids)].iterrows():
    cond_lookup[row['nct_id']].append(row['name'])

# [STEP 2] Interventions & Other Names
df_int = safe_load('interventions.txt', cols=['nct_id', 'id', 'intervention_type', 'name', 'description'])
df_int_others = safe_load('intervention_other_names.txt', cols=['nct_id', 'intervention_id', 'name'])
synonym_map = defaultdict(list)
for _, row in df_int_others.iterrows():
    synonym_map[row['intervention_id']].append(row['name'])

int_lookup = defaultdict(list)
for _, row in df_int[df_int['nct_id'].isin(target_ids)].iterrows():
    others = synonym_map.get(row['id'], [])
    other_str = f" (Aliases: {', '.join(others)})" if others else ""
    desc = day_zero_reconstructor(row['description'], "intervention") if pd.notna(row['description']) else "No description"
    int_lookup[row['nct_id']].append(f"NAME: {row['name']}{other_str} [{row['intervention_type']}]" + NL + f"DESC: {desc}")

# [STEP 3] Outcomes (Run 3 Requirement)
df_out1 = safe_load('outcomes.txt', cols=['nct_id', 'outcome_type', 'title', 'time_frame'])
df_out2 = safe_load('design_outcomes.txt', cols=['nct_id', 'outcome_type', 'measure', 'time_frame'])
df_out2 = df_out2.rename(columns={'measure': 'title'})
df_outcomes = pd.concat([df_out1, df_out2], ignore_index=True)

outcomes_lookup = defaultdict(list)
for _, out in df_outcomes[df_outcomes['nct_id'].isin(target_ids)].iterrows():
    if 'primary' in str(out['outcome_type']).lower():
        m = day_zero_reconstructor(str(out['title']), "outcome")
        t = day_zero_reconstructor(str(out['time_frame']), "timeframe")
        outcomes_lookup[out['nct_id']].append(f"TITLE: {m} | TIMEFRAME: {t}")

# [STEP 4] Summaries & Eligibility
df_sum = safe_load('brief_summaries.txt', cols=['nct_id', 'description'])
sum_lookup = {row['nct_id']: row['description'] for _, row in df_sum[df_sum['nct_id'].isin(target_ids)].iterrows()}

df_elig = safe_load('eligibilities.txt', cols=['nct_id', 'criteria'])
elig_lookup = {row['nct_id']: row['criteria'] for _, row in df_elig[df_elig['nct_id'].isin(target_ids)].iterrows()}

print("> Evidence Harvesting Complete.")

>>> Harvesting Enhanced Evidence...
> Evidence Harvesting Complete.


# **5. Context Assembly & Production Export**
The final step iterates through the cohort, retrieves the enhanced scientific evidence, and applies the Day Zero Sanitizer. The resulting molecular context blocks are saved for the Run 2 enrichment stage.

In [7]:
results = []
for nct_id in target_ids:
    s = studies_lookup.get(nct_id, {})
    r1 = run1_lookup.get(nct_id, {})

    start_date = s.get('start_date', '2000-01-01')
    start_year = str(pd.to_datetime(start_date).year)
    clean_title = day_zero_reconstructor(s.get('official_title', 'Unknown'), "title")
    conds = " | ".join(list(set(cond_lookup.get(nct_id, []))))
    ints_string = (NL + "---" + NL).join(int_lookup.get(nct_id, ["No intervention details"]))
    outcomes_string = (NL + "---" + NL).join(outcomes_lookup.get(nct_id, ["No primary outcomes listed"])[:10])
    clean_summary = day_zero_reconstructor(sum_lookup.get(nct_id, ""), "summary")
    clean_criteria = day_zero_reconstructor(elig_lookup.get(nct_id, ""), "criteria")

    context_body = f"""[TRIAL_START]
    [NCT_ID]: {nct_id}
    [START_YEAR]: {start_year}
    [PHASE]: {s.get('phase', 'N/A')}
    [ASSIGNED_INDICATION]: {r1.get('gbd_indication_name', 'Unknown')}
    [ASSIGNED_THERAPEUTIC_AREA]: {r1.get('therapeutic_area', 'Unclassified')}
    [OFFICIAL_TITLE]: {clean_title}

    [TRIAL_CONDITIONS_RAW]:
    {conds}

    [INTERVENTION_DETAILS_ENHANCED]:
    {ints_string}

    [PRIMARY_ENDPOINTS_DETAIL]:
    {outcomes_string}

    [PROTOCOL_SUMMARY]:
    {clean_summary[:4000]}...

    [ELIGIBILITY_CRITERIA_FULL]:
    {clean_criteria[:8000]}...
    [TRIAL_END]"""

    results.append({"nct_id": nct_id, "context": context_body})

pd.DataFrame(results).to_csv(os.path.join(DATA_PATH, 'llm_in_02.csv'), index=False)
print(f"> Success: Assembled Run 2+3 context for {len(results)} trials.")

> Success: Assembled Run 2+3 context for 29557 trials.


---
## **Next Step: Run Enrichment Orchestrator**
After generating the context CSV in this notebook, shift to your terminal and execute the following command to start the LLM processing stage:

```bash
python3 src/prep/llm_in_02_run.py
```